# Import Variables and Load Packages

In [ ]:
import uproot
#print("uproot version: ", uproot.__version__)
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from tqdm.notebook import tqdm
import pickle
from collections import Counter
from particle import Particle
import matplotlib as mpl

import awkward as ak

f = uproot.open('/Users/katherinepulido/Desktop/H-KPMTcalibration/HKPMTcalibration/LIGen395_Pos0_hits_flat.root')
print(f.keys())

print(f['photonTree'].keys())
f['hitsTree'].keys()



## Load Variables

In [ ]:
photon = ['event', 
          'startX', 
          'startY', 
          'startZ', 
          'endX', 
          'endY', 
          'endZ'
        ]

hits = ['event', 'subevent', 'pmt_id', 'charge', 'time', 'posX', 'posY', 'posZ']

vars = {}
vars2 = {}
vars.update(f["photonTree"].arrays(photon, library="np"))
vars2.update(f["hitsTree"].arrays(hits, library="np"))

for key, value in vars.items():
    print(f"{key}: {len(value)}")

for key, value in vars2.items():
    print(f"{key}: {len(value)}")
    
for col in vars:
    vars[col] = vars[col].tolist()
    
for col in vars2:
    vars2[col] = vars2[col].tolist()

ph_df = pd.DataFrame(vars)
hit_df = pd.DataFrame(vars2)

In [ ]:
#ph_df[ph_df["event"] == 9999]
#hit_df[hit_df["event"] == 9999]

#small_y_df = ph_df[abs(ph_df["startY"])<.0001]

moved_df = ph_df[
    (ph_df["startX"] != ph_df["endX"]) &
    (ph_df["startY"] != ph_df["endY"]) &
    (ph_df["startZ"] != ph_df["endZ"])
]

near_source_df = ph_df[ph_df["startX"]<40000]

moved_and_near_source_df = moved_df[abs(moved_df["startX"])<40000]

print(f'number of moved photons (start x,y,z != end x,y,z): {len(moved_df)}')
print(f'number of photons near source (startX<4000): {len(near_source_df)}')
print(f'number of photons that move and have startX<4000: {len(moved_and_near_source_df)}')

# can one photon hit on two pmts?

In [ ]:
# make a pmt dicitonary

pmt_dict = {
    row.pmt_id: (row.posX, row.posY, row.posZ)
    for row in hit_df[['pmt_id', 'posX', 'posY', 'posZ']]
        .drop_duplicates(subset='pmt_id')
        .itertuples(index=False)
}

pmt_dict[1234]

pmt_normals = {}
for row in hit_df[['pmt_id', 'posX', 'posY', 'posZ']].drop_duplicates(subset='pmt_id').itertuples(index=False):
    pmt_id, pmt_x, pmt_y, pmt_z = row.pmt_id, row.posX, row.posY, row.posZ

    # define normal vector
    if pmt_z == 3296.47119140625:
        normal_vector = np.array([0, 0, -1])
    elif pmt_z == -3296.47119140625:
        normal_vector = np.array([0, 0, 1])
    else:
        normal_vector = np.array([-pmt_x/radius, -pmt_y/radius, 0])

    # normalize
    normal_vector = normal_vector / np.linalg.norm(normal_vector)

    # store in separate dict
    pmt_normals[pmt_id] = normal_vector
    


# 3D Plotting

In [ ]:
# standard cylinder

ax = plt.figure(figsize=(12,12)).add_subplot(projection='3d')

mpl.rcParams['agg.path.chunksize'] =  4051901

# Prepare arrays x, y, z
zs = hit_df["posZ"][:20000]
xs = hit_df["posX"][:20000]
ys = hit_df["posY"][:20000]

xmin = -3242.76611328125
xmax = 3242.76611328125
ymin = -3242.76611328125
ymax = 3242.76611328125
zmin = -3296.47119140625
zmax = 3296.47119140625

print(f'x: {xmin, xmax}, y: {ymin, ymax}, z: {zmin, zmax}')

radius = 3242.76611328125
height = 2*3296.47119140625
z = np.linspace(-3296.47119140625, 3296.47119140625, 1000)
theta = np.linspace(0, 2*np.pi, 1000)
Theta, Zc = np.meshgrid(theta, z)
Xc = radius * np.cos(Theta)
Yc = radius * np.sin(Theta)

# Draw parameters
rstride = 20
cstride = 10
ax.plot_surface(Xc, Yc, Zc, alpha=0.1, rstride=rstride, cstride=cstride, color='grey')
ax.plot_surface(Xc, -Yc, Zc, alpha=0.1, rstride=rstride, cstride=cstride, color='grey')

ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.set_zlabel("Z")

ax.legend()

plt.show()

In [ ]:

def make_display(event_idx):    
    ax = plt.figure(figsize=(12,12)).add_subplot(projection='3d')

    #cylinder
    radius = 3242.76611328125
    z = np.linspace(-3296.47119140625, 3296.47119140625, 1000)
    theta = np.linspace(0, 2*np.pi, 1000)
    Theta, Zc = np.meshgrid(theta, z)
    Xc = radius * np.cos(Theta)
    Yc = radius * -np.sin(Theta)
    # cyl parameters
    rstride = 20
    cstride = 10
    ax.plot_surface(Xc, Yc, Zc, alpha=0.1, rstride=rstride, cstride=cstride, color='grey')
    ax.plot_surface(Xc, -Yc, Zc, alpha=0.1, rstride=rstride, cstride=cstride, color='grey')

    # add source positions
    source_positions = {
        0: (3220, 0, -2465.66),
        4: (3220, 0, -1643.78),
        8: (3220, 0, -821.88),
        12: (3220, 0, 0),
        16: (3220, 0, 821.888),
        20: (3220, 0, 1643.78),
        24: (3220, 0, 2465.66),
    }

    for key, (x, y, z) in source_positions.items():
        ax.scatter(x, y, z, label=f"Source Idx: {key}", s=100)

    event_ph_df = ph_df[ph_df["event"] == event_idx]
    event_hit_df = hit_df[hit_df["event"] == event_idx]

    # Apply filter
    y_df = event_ph_df
    #y_df = ph_df.sample(n=10000)

    # Only take 2000 rows
    #y_df = y_df.iloc[:50000]

    print(y_df.shape[0])


    # Start points
    #x_starts = np.full(y_df.shape[0], 3220)
    #y_starts = np.zeros(y_df.shape[0])
    #z_starts = np.full(y_df.shape[0], -2465.66)

    x_starts = y_df["startX"].values * 0.1
    y_starts = y_df["startY"].values * 0.1
    z_starts = y_df["startZ"].values * 0.1

    # End points
    x_ends = y_df["endX"].values * 0.1
    y_ends = y_df["endY"].values * 0.1
    z_ends = y_df["endZ"].values * 0.1
        
    pmt_xs = event_hit_df['posX']
    pmt_ys = event_hit_df['posY']
    pmt_zs = event_hit_df['posZ']
    pmt_charges = event_hit_df['charge']
    #event_hit_df['posX']

    # Plot line segments
    for xs, ys, zs, xe, ye, ze in zip(x_starts, y_starts, z_starts, x_ends, y_ends, z_ends):
        ax.plot([xs, xe], [ys, ye], [zs, ze], color='blue', alpha=0.2)
        for pmtx, pmty, pmtz, pmte, in zip(pmt_xs, pmt_ys, pmt_zs, pmt_charges):
            if np.linalg.norm(np.array([xe, ye, ze]) - np.array([pmtx, pmty, pmtz])) < 26.95:
                ax.scatter(pmtx, pmty, pmtz, color='red')
                
    pmt_hit_df = hit_df[hit_df["pmt_id"]==19343]

    #x1 = pmt_hit_df["posX"].iloc[0] 
    #y1 = pmt_hit_df["posY"].iloc[0] 
    #z1 = pmt_hit_df["posZ"].iloc[0] 
 
    pmt_410 = hit_df[hit_df["pmt_id"] == 410]
    
    x1 = pmt_410["posX"]
    y1 = pmt_410["posY"]
    z1 = pmt_410["posZ"]
    
    
    
    ax.scatter(x1, y1, z1, color='purple', label='pmt 410', marker="o", s=250)

    ax.set_xlabel("X (cm)")
    ax.set_ylabel("Y (cm)")
    ax.set_zlabel("Z (cm)")

    ax.legend()
    plt.title(f'Photon Tracks with PMT Hits for Event {event_idx}')

    plt.show()

make_display(1)

# make version for one PMT with angular dependence
# look for PMT with highest amt of hits
# hit count ratio as a function of incident angle



In [ ]:
pmt_hit_df = hit_df[hit_df["pmt_id"]==13007]

x1 = pmt_hit_df["posX"].iloc[0]
y1 = pmt_hit_df["posY"].iloc[0]
z1 = pmt_hit_df["posZ"].iloc[0]

print(x1, y1, z1)

In [ ]:
# sai's code to compute nearest neighbors

import numpy as np
from scipy.spatial import cKDTree

def cartesian_to_cylindrical(x, y, z):
    """Convert Cartesian coordinates to cylindrical coordinates (r, phi, z)."""
    r = np.sqrt(x**2 + y**2)
    phi = np.arctan2(y, x)
    return r, phi, z

def get_cylinder_neighbours(target_pmt_id, pmt_positions_df, k=6, barrel_radius=3220):
    """
    Find nearest neighbours of a target PMT, accounting for cylinder geometry.
    
    Parameters
    ----------
    target_pmt_id : int
        PMT ID to find neighbours for.
    pmt_positions_df : pd.DataFrame
        Must contain columns: 'posX','posY','posZ','pmt_id'.
    k : int
        Number of PMTs to query (includes the target itself).
    barrel_radius : float
        Cylinder radius for barrel PMTs.
    
    Returns
    -------
    neighbour_ids : np.ndarray
        Array of PMT IDs of the nearest neighbours (excluding target).
    """
    
    # Find the target PMT
    target_row = pmt_positions_df[pmt_positions_df['pmt_id'] == target_pmt_id]
    if target_row.empty:
        raise ValueError(f"Target PMT {target_pmt_id} not found.")
    target_idx = target_row.index[0]

    # Determine if barrel or endcap
    x, y, z = target_row[['posX','posY','posZ']].values[0]
    r = np.sqrt(x**2 + y**2)
    if np.isclose(r, barrel_radius, atol=50):
        detector_type = 'barrel'
    else:
        detector_type = 'endcap'

    # Build surface-aware points
    if detector_type == 'barrel':
        _, phi, z_all = cartesian_to_cylindrical(
            pmt_positions_df['posX'].values,
            pmt_positions_df['posY'].values,
            pmt_positions_df['posZ'].values
        )
        points = np.column_stack([barrel_radius * phi, z_all])
    else:  # endcap
        points = pmt_positions_df[['posX','posY']].values

    # Build KDTree
    tree = cKDTree(points)

    # Query k nearest neighbours
    target_point = points[target_idx]
    distances, neighbour_indices = tree.query(target_point, k=k)
    neighbour_indices = neighbour_indices[1:]  # exclude target itself

    neighbour_ids = pmt_positions_df.iloc[neighbour_indices]['pmt_id'].values
    return neighbour_ids



target_pmt_id = 8003
neighbours = get_cylinder_neighbours(target_pmt_id, pmt_positions_df, k=6, barrel_radius=3220)

## Incident Angle Studies

In [ ]:
#hit_df["posX"].mode()[0]
#x_common_df = hit_df[hit_df["posX"]==1341.4000244140625]
#y_common_df = x_common_df[x_common_df["posY"]==x_common_df["posY"].mode()[0]]
#z_common_df = y_common_df[y_common_df["posY"]==y_common_df["posY"].mode()[0]]

# ratio of hits based on azimuthal angle (x-axis) and distance from center of PMT

#common_df = hit_df[hit_df["pmt_id"]==hit_df["pmt_id"].mode()[0]]

x_ends = ph_df["endX"].values * 0.1
y_ends = ph_df["endY"].values * 0.1
z_ends = ph_df["endZ"].values * 0.1

x_starts = ph_df["startX"].values * 0.1
y_starts = ph_df["startY"].values * 0.1
z_starts = ph_df["startZ"].values * 0.1

def get_ang_pos(pmt_id):
    pmt_hit_df = hit_df[hit_df["pmt_id"] == pmt_id]
    pmt_x, pmt_y, pmt_z = pmt_hit_df["posX"].iloc[0], pmt_hit_df["posY"].iloc[0], pmt_hit_df["posZ"].iloc[0] 
    
    normal_vector = []

    # define normal vector of PMT
    if pmt_z == 3296.47119140625:
        normal_vector = np.array([0, 0,-1])
    elif pmt_z == -3296.47119140625:
        normal_vector = np.array([0, 0, 1])
    else:
        normal_vector = np.array([-pmt_x/radius, -pmt_y/radius, 0])

    # normalize normal vector
    normal_vector = normal_vector / np.linalg.norm(normal_vector)
    
    # vector from photon end to PMT center
    dx = x_ends - pmt_x
    dy = y_ends - pmt_y
    dz = z_ends - pmt_z

    center_dist = np.sqrt(dx**2 + dy**2 + dz**2)

    mask = center_dist < 50
    
    dist_to_source = np.sqrt((x_starts - pmt_x)**2 + (y_starts - pmt_y)**2 + (z_starts - pmt_z)**2)

    # photon direction
    ph_vx = x_starts - x_ends
    ph_vy = y_starts - y_ends
    ph_vz = z_starts - z_ends

    ph_norm = np.sqrt(ph_vx**2 + ph_vy**2 + ph_vz**2)

    ph_vx /= ph_norm
    ph_vy /= ph_norm
    ph_vz /= ph_norm

    cos_theta = (
        normal_vector[0]*ph_vx +
        normal_vector[1]*ph_vy +
        normal_vector[2]*ph_vz
    )

    theta = np.degrees(np.arccos(np.clip(cos_theta, -1, 1)))
    
    weird_mask = mask & (theta>175)
    
    idx = np.where(weird_mask)[0]


    return theta[mask], dist_to_source[mask], idx

# find theta, distance for every pmt

#from tqdm import tqdm

thetas = []
center_distances = []
all_idxs = []

unique_pmts = hit_df['pmt_id'].unique()

for id in tqdm(unique_pmts):
    theta, dist, idx = get_ang_pos(id)
    thetas.append(theta)
    center_distances.append(dist)
    all_idxs.append(idx)

all_idxs = np.concatenate(all_idxs)

selected_starts = np.stack([
    x_starts[all_idxs],
    y_starts[all_idxs],
    z_starts[all_idxs]
], axis=1)

selected_ends = np.stack([
    x_ends[all_idxs],
    y_ends[all_idxs],
    z_ends[all_idxs]
], axis=1)
    


# plot thetas vs center_distances on histogram

plt.figure(figsize=(12,8))

plt.hist2d(np.concatenate(center_distances), np.concatenate(thetas), bins=100)

plt.xlabel("Distance from PMT to Source (cm)")
plt.title("2D histogram comparing incident angle and distance from photon source")
plt.ylabel("Theta (degrees)")
plt.colorbar(label="Counts")




    
    
    
    # if close to PMT, calculate azimuthal angle and distance to center of PMT
    #for xs, ys, zs, xe, ye, ze in zip(x_starts, y_starts, z_starts, x_ends, y_ends, z_ends):
    #    if np.linalg.norm(np.array([xe, ye, ze]) - np.array([pmt_x, pmt_y, pmt_z])) < 30:
            
            
            
            
        
    #    ax.plot([xs, xe], [ys, ye], [zs, ze], color='blue', alpha=0.2)
    #    for pmtx, pmty, pmtz, pmte, in zip(pmt_xs, pmt_ys, pmt_zs, pmt_charges):
    #        if np.linalg.norm(np.array([xe, ye, ze]) - np.array([pmtx, pmty, pmtz])) < 26.9:
    #            ax.scatter(pmtx, pmty, pmtz, color='red')
            
            







In [ ]:
plt.figure(figsize=(12,8))

plt.hist2d(np.concatenate(center_distances), np.concatenate(thetas), bins=100)

plt.xlabel("Distance from PMT to Source (cm)")
plt.title("2D histogram comparing incident angle and distance from photon source")
plt.ylabel("Theta (degrees)")
plt.colorbar(label="Counts")

In [ ]:
# study large angle

unique_pmts = hit_df['pmt_id'].unique()[:1000]

for id in tqdm(unique_pmts):
    theta, dist, idx = get_ang_pos(id)
    thetas.append(theta)
    center_distances.append(dist)
    all_idxs.append(idx)

all_idxs = np.concatenate(all_idxs)

selected_starts = np.stack([
    x_starts[all_idxs],
    y_starts[all_idxs],
    z_starts[all_idxs]
], axis=1)

selected_ends = np.stack([
    x_ends[all_idxs],
    y_ends[all_idxs],
    z_ends[all_idxs]
], axis=1)


    

In [ ]:

    ax = plt.figure(figsize=(12,12)).add_subplot(projection='3d')

    #cylinder
    radius = 3242.76611328125
    z = np.linspace(-3296.47119140625, 3296.47119140625, 1000)
    theta = np.linspace(0, 2*np.pi, 1000)
    Theta, Zc = np.meshgrid(theta, z)
    Xc = radius * np.cos(Theta)
    Yc = radius * np.sin(Theta)
    # cyl parameters
    rstride = 20
    cstride = 10
    ax.plot_surface(Xc, Yc, Zc, alpha=0.1, rstride=rstride, cstride=cstride, color='grey')
    ax.plot_surface(Xc, -Yc, Zc, alpha=0.1, rstride=rstride, cstride=cstride, color='grey')

    # add source positions
    source_positions = {
        0: (3220, 0, -2465.66),
        4: (3220, 0, -1643.78),
        8: (3220, 0, -821.88),
        12: (3220, 0, 0),
        16: (3220, 0, 821.888),
        20: (3220, 0, 1643.78),
        24: (3220, 0, 2465.66),
    }

    for key, (x, y, z) in source_positions.items():
        ax.scatter(x, y, z, label=f"Source Idx: {key}", s=100)

    #event_ph_df = ph_df[ph_df["event"] == event_idx]
    #vent_hit_df = hit_df[hit_df["event"] == event_idx]

    # Apply filter
    #y_df = event_ph_df
    #y_df = ph_df.sample(n=10000)

    # Only take 2000 rows
    #y_df = y_df.iloc[:50000]

    #print(y_df.shape[0])


    # Start points
    #x_starts = np.full(y_df.shape[0], 3220)
    #y_starts = np.zeros(y_df.shape[0])
    #z_starts = np.full(y_df.shape[0], -2465.66)
        
    pmt_xs = hit_df['posX']
    pmt_ys = hit_df['posY']
    pmt_zs = hit_df['posZ']
    pmt_charges = hit_df['charge']
    #event_hit_df['posX']

    # Plot line segments
    for start, end in zip(selected_starts, selected_ends):
        xs, ys, zs = start
        xe, ye, ze = end

        print(xs, ys, zs, xe, ye, ze)
        ax.plot([xs, xe], [ys, ye], [zs, ze], color='blue', alpha=0.2)
        #for pmtx, pmty, pmtz, pmte, in zip(pmt_xs, pmt_ys, pmt_zs, pmt_charges):
        #    if np.linalg.norm(np.array([xe, ye, ze]) - np.array([pmtx, pmty, pmtz])) < 26.95:
        #        ax.scatter(pmtx, pmty, pmtz, color='red')
                
    pmt_hit_df = hit_df[hit_df["pmt_id"]==19343]

    x1 = pmt_hit_df["posX"].iloc[0] 
    y1 = pmt_hit_df["posY"].iloc[0] 
    z1 = pmt_hit_df["posZ"].iloc[0] 
    
    #ax.scatter(x1, y1, z1, color='yellow', label='pmt', s=500)

    ax.set_xlabel("X (cm)")
    ax.set_ylabel("Y (cm)")
    ax.set_zlabel("Z (cm)")

    ax.legend()
    plt.title(f'Photon Tracks with PMT Hits for Large Angles')

    plt.show()

In [ ]:
theta1, _, _ = get_ang_pos(3004)

# plot distribution of theta for a given event

index = max(range(len(thetas)), key=lambda i: np.std(thetas[i]))
largest_std_list = thetas[index]

plt.figure(figsize=(12,8))

plt.hist(largest_std_list, bins=30)

plt.xlabel("Angle (degrees)")
plt.title("Distribution of Theta for PMT 410")

plt.figure(figsize=(12,8))

plt.hist(theta1, bins=10)

plt.xlabel("Angle (degrees)")
plt.title("Distribution of Zenith Angle for PMT 3004")

print(index)



## Intuitive Interfaces

In [ ]:
# just the circle and the cylinder

import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Rectangle

fig, ax = plt.subplots(figsize=(10, 12))

# rectangle centered
rect = Rectangle((-np.pi*radius, -height/2), 2*np.pi*radius, height, edgecolor='black', facecolor='none')
ax.add_patch(rect)

# circles
top_circle = Circle((0, height/2 + radius), radius, edgecolor='black', facecolor='none')
ax.add_patch(top_circle)
bottom_circle = Circle((0, (-height/2)-radius), radius, edgecolor='black', facecolor='none')
ax.add_patch(bottom_circle)

ax.set_xlim(-1.2*np.pi*radius, 1.2*np.pi*radius)
ax.set_ylim(-1.2*radius - height, 1.2*radius + height)
ax.set_aspect('equal')

ax.set_xlabel("Distance around barrel, centered about the positive x-axis (cm)")
ax.set_ylabel("Height (cm)")

plt.show()

In [ ]:
# get nearest neighbors

pmt_lookup = (
    hit_df[['pmt_id', 'posX', 'posY', 'posZ']]
    .drop_duplicates(subset='pmt_id')
    .set_index('pmt_id')
)

all_pmt_ids = pmt_lookup.index.values
positions = pmt_lookup[['posX','posY','posZ']].values

hit_counts = hit_df['pmt_id'].value_counts()

def get_nearest_pmts(pmt_id, k):
    if pmt_id not in pmt_lookup.index:
        raise ValueError(f"PMT {pmt_id} not found")

    target = pmt_lookup.loc[pmt_id].values

    # compute distances
    dists = np.linalg.norm(positions - target, axis=1)
    mask = all_pmt_ids != pmt_id

    nearest_idx = np.argsort(dists[mask])[:k]
    nearest_pmts = all_pmt_ids[mask][nearest_idx]

    return nearest_pmts

hit_counts = hit_df['pmt_id'].value_counts()



## Load Variables

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Circle
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from matplotlib.lines import Line2D

def get_pmt_normal(pmt_id):
    """Return the normal vector for a PMT, with top/bottom special cases."""
    x, y, z = pmt_dict[pmt_id]
    
    # Top / bottom check
    if np.isclose(z, 3296.47119140625):
        return np.array([0, 0, -1])
    elif np.isclose(z, -3296.47119140625):
        return np.array([0, 0, 1])
    
    # Otherwise, use the stored normal
    n = np.array(pmt_normals[pmt_id])
    return n / np.linalg.norm(n)

def pmt_2d_projection(pmt_ids, pmt_info, size, info_label, dot_size, show_vals=False):
    fig, (ax_barrel, ax_persp) = plt.subplots(1, 2, figsize=(18, 10))
    
    # some vars that are globally relevant
    center_pmt = pmt_ids[0]
    n = get_pmt_normal(center_pmt)
    cent_pos = np.array(pmt_dict[center_pmt])
    cent_x, cent_y, cent_z = cent_pos
    
    
    # full barrel plot
    rect = Rectangle((-np.pi*radius, -height/2), 2*np.pi*radius, height, edgecolor='black', facecolor='none')
    ax_barrel.add_patch(rect)
    top_circle = Circle((0, height/2 + radius), radius, edgecolor='black', facecolor='none')
    ax_barrel.add_patch(top_circle)
    bottom_circle = Circle((0, (-height/2)-radius), radius, edgecolor='black', facecolor='none')
    ax_barrel.add_patch(bottom_circle)
    
    
    ax_barrel.set_xlim(-1.2*np.pi*radius, 1.2*np.pi*radius)
    ax_barrel.set_ylim(-1.2*radius - height, 1.2*radius + height)
    ax_barrel.set_aspect('equal')
    ax_barrel.set_xlabel("Distance around barrel (cm)")
    ax_barrel.set_ylabel("Height (cm)")
    ax_barrel.set_title("Full Cylinder View")

    # plot center on barrel plot, make sure to check if on top/bottom
    if np.isclose(cent_z, 3296.47119140625):
        # Top PMT
        star_x = cent_x          # x-coordinate in barrel plot
        star_z = cent_y + radius + height/2  # place on top circle
    elif np.isclose(cent_z, -3296.47119140625):
        # Bottom PMT
        star_x = cent_x
        star_z = cent_y - radius - height/2  # place on bottom circle
    else:
        # Regular PMT
        star_x = cent_x * np.pi
        star_z = cent_z

    # Plot center PMT as a star
    ax_barrel.scatter(
        star_x, star_z,
        marker='*',
        color='#FF7F0E',  # bright orange
        s=300,
        edgecolors='k',
        label=f'Center PMT: {center_pmt}'
    )
    
    ax_barrel.legend(loc='upper right')
    
    # red square highlighting region of interest
    
    center_pmt = pmt_ids[0]
    n = get_pmt_normal(center_pmt)
    cent_pos = np.array(pmt_dict[center_pmt])
    # select whether we are working with top/bottom once again
    if np.allclose(n, [0, 0, -1]):
        # top
        roi = Rectangle(
            (cent_pos[0] - size/2, (cent_pos[1] + radius + height/2) - size/2),
            size, size,
            facecolor='red', alpha=0.2
        )
        ax_barrel.add_patch(roi)
        
    elif np.allclose(n, [0, 0, 1]):
        # bottom
        roi = Rectangle(
            (cent_pos[0] - size/2, (cent_pos[1] - radius - height/2) - size/2),
            size, size,
            facecolor='red', alpha=0.2
        )
        ax_barrel.add_patch(roi)
        
    else:
        # barrel
        roi = Rectangle(
            (cent_pos[0]*np.pi - size/2, cent_pos[2] - size/2),
            size, size,
            facecolor='red', alpha=0.2
        )
        ax_barrel.add_patch(roi)
    
    # PMT perspective plot
    
    # create orthonormal basis for projection
    if np.allclose(n, [0,0,1]) or np.allclose(n, [0,0,-1]):
        # Vertical normal: use x and y axes
        u = np.array([1,0,0])
        v = np.array([0,1,0])
    else:
        # General case
        u = np.cross([0,0,1], n)  # u is perpendicular to both z and n (phi hat)
        u /= np.linalg.norm(u)      # normalize, just in case
        v = np.cross(n, u)          # v is perpendicular to n and u, (should be z hat)
    
    info_array = np.array(pmt_info, dtype=float)
    center_value = info_array[0]

    # Avoid division by zero
    if center_value == 0:
        center_value = 1.0

    # Normalize relative to center
    info_array = info_array / center_value

    # Robust normalization centered at 1
    vmin = info_array.min()
    vmax = info_array.max()

    # Ensure valid ordering for TwoSlopeNorm
    if vmin == vmax:
        vmin -= 1e-6
        vmax += 1e-6

    norm = mcolors.TwoSlopeNorm(vmin=vmin, vcenter=1.0, vmax=vmax)

    # Colormap
    cmap = cm.get_cmap('viridis')

    for i, (pmt_id, info) in enumerate(zip(pmt_ids, info_array)):
        pos = np.array(pmt_dict[pmt_id]) - cent_pos
        x_proj = np.dot(pos, u)
        y_proj = np.dot(pos, v)

        color = cmap(norm(info))

        sc = ax_persp.scatter(
            x_proj, y_proj,
            c=[color],
            s=dot_size,
            label=info_label if i == 0 else None,
            zorder=2
        )

        if show_vals:
            text_color = 'white' if norm(info) > 0.5 else 'black'
            ax_persp.text(
                x_proj, y_proj,
                f"{info:.2f}",
                ha='center', va='center',
                fontsize=8,
                color=text_color,
                zorder=3
            )
        #print(f"PMT {pmt_id}: info={info}, projected=({x_proj:.1f},{y_proj:.1f}), marker_size={marker_size:.1f}")
    legend_dot = Line2D(
        [0], [0], 
        marker='o', 
        color='w',                  # line itself invisible
        markerfacecolor='#21908C',  # fill color of dot
        markeredgecolor='#21908C',  # optional: edge color
        markersize=8, 
        label=info_label)
    ax_persp.legend(handles=[legend_dot], loc='upper right')
    
    ax_persp.set_xlim(-size/2, size/2)
    ax_persp.set_ylim(-size/2, size/2)
    ax_persp.set_xlabel("Distance Projected Along u (cm)")
    ax_persp.set_ylabel("Distance Projected Along v (cm)")
    ax_persp.set_aspect('equal')
    ax_persp.set_title("Perspective View Along PMT Normal")
    ax_persp.tick_params(axis='both', which='both', labelsize=10)
    plt.colorbar(sc, ax=ax_persp, label='Relative Charge')
    fig.tight_layout()
    plt.show()
    

In [ ]:

center_pmt = 410
num = 10
pmt_nn_ids = get_nearest_pmts(center_pmt, num)

pmt_ids = np.concatenate(([center_pmt], pmt_nn_ids))

pmt_hit_info = [get_hit_count(p) for p in pmt_ids]
#pmt_charge_info = [get_charge(p) for p in pmt_ids]

def get_angle(pmt_id):
    thetas = get_ang_pos(pmt_id)  # keep as a list
    # flatten in case it returns nested lists
    flat_thetas = []
    for t in thetas:
        if isinstance(t, (list, np.ndarray)):
            flat_thetas.extend(t)
        else:
            flat_thetas.append(t)
    return np.mean(flat_thetas)

pmt_angle_info = [get_angle(p) for p in pmt_ids]

### TRY NOT TO RESET KERNEL, ACCIDENTALLY DELETED DEFINITION OF CHARGE AND HIT COUNT FUNCTIONS

In [ ]:
#pmt_2d_projection(pmt_ids, pmt_info, 6000, "Total Charge", 50)

#pmt_2d_projection(pmt_ids, pmt_info, 6000, "Values Relative to Center", 20)

#pmt_2d_projection(pmt_ids, pmt_hit_info, 400, "Values Relative to Center", 2000, True)
#pmt_2d_projection(pmt_ids, pmt_charge_info, 600, "Values Relative to Center", 5000, True)

pmt_2d_projection(pmt_ids, pmt_angle_info, 400, "Values Relative to Center", 2000, False)

# look at hits from the PMT perspective